# Explore feed decisions

Loads `data/decisions.jsonl` (written by the bot) and helps you pick thresholds for `classifier.json`.

In [ ]:
from decisions import load_decisions, load_classifier, should_hide, ROOT
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_colwidth', 120)
df = load_decisions()
classifier = load_classifier()
print(f'{len(df)} decisions, {df.post_id.nunique()} unique posts')
df.tail()

## How often does it hide?

In [ ]:
df.groupby(['classifier_version', 'dry_run'])[['hide', 'acted']].mean().round(2)

## Score distributions

One histogram per numeric answer (`noul` / `score`). A good question splits posts into two clear humps.

In [ ]:
numeric = [c for c in df.columns if c.endswith(('.noul', '.score'))]
fig, axes = plt.subplots(1, len(numeric), figsize=(5 * len(numeric), 3), squeeze=False)
for ax, col in zip(axes[0], numeric):
    df[col].plot.hist(bins=20, ax=ax, title=col)
plt.tight_layout()

## Threshold sweep

What share of posts would be hidden at each cutoff for the first `noul` question?

In [ ]:
col = next(c for c in df.columns if c.endswith('.noul'))
sweep = pd.DataFrame({'threshold': [i / 20 for i in range(21)]})
sweep['hidden_share'] = [(df[col] >= t).mean() for t in sweep.threshold]
sweep.plot(x='threshold', y='hidden_share', marker='o', title=col, legend=False)

## Borderline posts

Read these to decide which side of the line they belong on.

In [ ]:
df[df[col].between(0.4, 0.8)].sort_values(col)[['author', 'text', col, 'hide']]

## Hand labels

Mark posts yourself (`True` = should hide), then check the rule against them. Labels are saved to `data/labels.csv`.

In [ ]:
labels_path = ROOT / 'data' / 'labels.csv'
labels = pd.read_csv(labels_path, dtype={'post_id': str}) if labels_path.exists() else pd.DataFrame(columns=['post_id', 'should_hide'])

# Example: labels.loc[len(labels)] = ['1234567890', True]; labels.to_csv(labels_path, index=False)

labeled = df.drop_duplicates('post_id', keep='last').merge(labels, on='post_id')
if len(labeled):
    print(pd.crosstab(labeled.should_hide, labeled.hide, rownames=['you'], colnames=['classifier']))

## Compare a replay

After `uv run --env-file ../.env replay.py`, load the newest replay file and see what changed.

In [ ]:
replays = sorted((ROOT / 'data').glob('replay-*.jsonl'))
if replays:
    new = load_decisions(replays[-1])
    old = df.drop_duplicates('post_id', keep='last')
    both = old.merge(new, on='post_id', suffixes=('_old', '_new'))
    print(replays[-1].name)
    display(both[both.hide_old != both.hide_new][['author_old', 'text_old', 'hide_old', 'hide_new']])